# MuJoCo: Your First Physics Simulation

**Goal:** Build a small scene, simulate a bouncing ball, and visualize the results. Works in **Binder** and **Google Colab** using offscreen rendering (no desktop GUI needed).

MuJoCo models are often written in **MJCF**, an XML format describing bodies, joints, geometry, and physical properties.

## 1. Install dependencies

**Colab:** Run the following cell once. **Binder:** The same dependencies are preinstalled by `requirements.txt`, so you can skip this cell. The installation cell is safe to rerun but may require a kernel restart after changing package versions.

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec("mujoco") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mujoco>=3.2,<4", "imageio>=2.34", "pillow>=10", "matplotlib>=3.8"])
print("Dependencies ready")

## 2. Create a scene

The ball is a free body (six degrees of freedom). The ground is fixed. `timestep` controls the physics step, while `gravity` acts on the ball.

In [ ]:
import os
# Must be set before importing mujoco to use software rendering on Linux hosts.
os.environ.setdefault("MUJOCO_GL", "osmesa")
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

XML = """
<mujoco model="bouncing ball">
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <visual><global offwidth="640" offheight="480"/></visual>
  <worldbody>
    <light pos="0 -2 4" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 0.1" rgba="0.82 0.85 0.9 1"/>
    <body name="ball" pos="0 0 1.5">
      <freejoint/>
      <geom name="sphere" type="sphere" size="0.15" mass="1" rgba="0.95 0.35 0.15 1" friction="0.6 0.01 0.001"/>
    </body>
    <camera name="side" pos="2.8 -4 2.1" xyaxes="0.82 0.57 0 -0.20 0.29 0.94"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)
print(f"Bodies: {model.nbody}; time step: {model.opt.timestep} s")

## 3. Run the physics simulation

`mj_step` advances physics by one timestep. We store the ball's height and a few frames for a short animation.

In [ ]:
duration = 3.0
steps = int(duration / model.opt.timestep)
ball_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "ball")
times, heights = [], []
frames = []
# Offscreen rendering via OSMesa on Binder/Linux, or EGL if configured externally.
try:
    renderer = mujoco.Renderer(model, height=360, width=480)
except Exception as exc:
    renderer = None
    print("Rendering unavailable in this environment:", exc)
for i in range(steps):
    mujoco.mj_step(model, data)
    times.append(data.time)
    heights.append(data.xpos[ball_id, 2])
    if renderer is not None and i % 12 == 0:
        renderer.update_scene(data, camera="side")
        frames.append(renderer.render().copy())
if renderer is not None:
    renderer.close()
plt.figure(figsize=(8, 3))
plt.plot(times, heights)
plt.xlabel("Time (s)"); plt.ylabel("Ball center height (m)")
plt.title("Ball height over time"); plt.grid(alpha=0.3)
plt.show()

## 4. Display the animation inline

This uses an embedded GIF, so it does not need a separate desktop viewer.

In [ ]:
if frames:
    import io
    from PIL import Image
    from IPython.display import Image as IPyImage
    buf = io.BytesIO()
    Image.fromarray(frames[0]).save(
        buf, format="GIF", save_all=True,
        append_images=[Image.fromarray(f) for f in frames[1:]],
        duration=60, loop=0, optimize=False)
    display(IPyImage(data=buf.getvalue()))
else:
    print("Animation skipped; the height plot above still shows the simulation.")

## 5. Experiment: change gravity and contact bounce

MuJoCo's `solref` controls contact softness and damping, **not** a literal coefficient of restitution. Compare height traces under different gravity settings, then try editing the contact parameters in the XML.

In [ ]:
def simulate_gravity(g, duration=2.0):
    m = mujoco.MjModel.from_xml_string(XML)
    m.opt.gravity[2] = -g
    d = mujoco.MjData(m)
    bid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, "ball")
    t, z = [], []
    for _ in range(int(duration / m.opt.timestep)):
        mujoco.mj_step(m, d)
        t.append(d.time); z.append(d.xpos[bid, 2])
    return t, z

for gravity in [1.62, 3.71, 9.81]:
    t, z = simulate_gravity(gravity)
    plt.plot(t, z, label=f"g = {gravity} m/s²")
plt.xlabel("Time (s)"); plt.ylabel("Ball center height (m)")
plt.title("Gravity changes the motion")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Try it yourself

1. Change the sphere's `mass` from `1` to `5`. Does free-fall acceleration change?
2. Change its initial `pos` from `0 0 1.5` to `0 0 2.5`.
3. Change the XML `gravity` to `0 0 -1.62` (Moon gravity).
4. Add a second sphere with a new body and free joint.

**Note:** Contact friction affects sliding/rolling more than a purely vertical drop. For a friction experiment, add horizontal initial velocity or an inclined plane.